# Step 1c: Cohere Transcribe and Whisper large-v3-turbo-German (GPU)

Kernel: **NeMo ASR** (`~/nemoenv`, see `../../environments/README.md`). The notebook needs a GPU. Run it after steps 1a and 1b, with the same `MEETING` and `AUDIO` in the first line of cell 1.

The notebook transcribes the speaker turns of the recording with two more recognizers. `protokoll.py` reads both transcripts with `--also-transcript`: two of the three recognizers must then agree on each handover name, and words on which they disagree go to the list of names to check.

**Reads**, relative to `pipeline/` (cell 1 changes to that folder): `outputs/<MEETING>/<AUDIO file name without extension>_16k_mono.wav` from step 1a, and `transcript_canary_ts.json` and `diarization_hyp.rttm` from step 1b. Cell 1 stops if one of them is missing.

**Clips.** Cell 1 builds the speaker turns from the Canary words with the same functions, in the same order, as `transcript_repair.repair()`, but without `verify_short()`. A few turns can therefore differ from the repaired main transcript. It adds the stretches without Canary words that `transcript_repair.find_gaps()` finds. A turn clip runs from its first word to 0.2 s after its last word. A gap clip runs from the end of the word before the gap to 0.2 s after the start of the word after it. `asr_chunks.split()` cuts a clip longer than 27 s at its quietest points, and every piece then gets 0.5 s of silence at both ends, so no piece is longer than 28 s.

**Writes** into `outputs/<MEETING>/`:
- `transcript_speakers_whisper_turbo_de.txt` (cell 2) and `transcript_speakers_cohere.txt` (cell 3), with lines `[HH:MM:SS] SPEAKER_NN: text`. A turn keeps its diarized speaker label. The text of a gap is shared among the speakers that the diarization finds in it.
- `step1c_segments.json` (cells 2 and 3): every clip with its start, its end and the texts decoded since cell 1 last ran.

**Order.** Run cells 1 to 4 in order. Cell 2 runs Whisper large-v3-turbo-German. Cell 3 runs Cohere Transcribe. Both load their model as `../../recognizers/NB_v10_asr.ipynb` does, and cell 3 has the same three transformers 5 fixes. Cell 4 checks that both transcripts exist and prints the two `--also-transcript` options for the `protokoll.py` command.

**If the kernel restarts**, run cell 1 again, then the cell that did not finish. `step1c_segments.json` then holds only the texts decoded after the restart. To have both texts in it, run cells 2 and 3 again. If a cell stops with an error and the kernel keeps running, run that cell again.

In [ ]:
MEETING, AUDIO = "april27", "data/Team-Meeting_2026-04-27.flac"

# 1. Speaker turns, gap stretches and the clips to decode
import os, sys, json, gc, tempfile
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "cluster" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
from thesis_pipeline.env import load_env
load_env()  # HF_TOKEN from pipeline/.env: Cohere Transcribe is a gated model

import numpy as np
import soundfile as sf
import torch
import transformers
import asr_chunks
from transcript_repair import load_words, load_rttm, find_gaps, gap_turns
from speaker_turns_v11 import sentence_vote, runs, absorb_short, split_long, hhmmss

assert torch.cuda.is_available(), "no GPU visible: use the 'NeMo ASR' kernel on a GPU server"
DEVICE = "cuda"
DTYPE = torch.bfloat16
DTYPE_KW = {"dtype": DTYPE} if int(transformers.__version__.split(".")[0]) >= 5 else {"torch_dtype": DTYPE}

out = Path("outputs") / MEETING
wav = out / (Path(AUDIO).stem + "_16k_mono.wav")
for p in (wav, out / "transcript_canary_ts.json", out / "diarization_hyp.rttm"):
    assert p.exists(), f"{p} is missing: run steps 1a and 1b for {MEETING} first"
y, sr = sf.read(str(wav), dtype="float32")
assert sr == 16000 and y.ndim == 1, f"{wav}: expected 16 kHz mono, got {sr} Hz, shape {y.shape}"

# Same steps and order as transcript_repair.repair(), without verify_short()
words = sentence_vote(load_words(out / "transcript_canary_ts.json"))
turns = [t for r in absorb_short(runs(words)) for t in split_long(r)]   # (speaker, start, end, Canary text)
rttm = load_rttm(out / "diarization_hyp.rttm")
gaps = find_gaps(words, rttm)                                         # stretches without Canary words

END_S, PAD_S = 0.2, 0.5
pad = np.zeros(int(PAD_S * sr), dtype=np.float32)

def clip(s, e):
    a, b = max(0, int(s * sr)), min(len(y), int((e + END_S) * sr))
    return y[a:b]

units = ([{"kind": "turn", "speaker": spk, "start": s, "end": e} for spk, s, e, _ in turns]
         + [{"kind": "gap", "gap": g, "start": g["start_s"], "end": g["end_s"]} for g in gaps])
pieces = [(i, np.concatenate([pad, x, pad])) for i, u in enumerate(units)
          for x in asr_chunks.split(clip(u["start"], u["end"]), sr, max_s=asr_chunks.MAX_S - 2 * PAD_S)]

def save(system, piece_texts):
    """Join the piece texts of each clip and write transcript_speakers_<system>.txt and step1c_segments.json."""
    per_unit = [[] for _ in units]
    for (i, _), t in zip(pieces, piece_texts):
        per_unit[i].append((t or "").strip())
    lines = []
    for u, texts in zip(units, per_unit):
        u[system] = text = asr_chunks.join(texts)
        if not text:
            continue
        if u["kind"] == "turn":
            lines.append((u["start"], u["speaker"], text))
        else:
            lines.extend(gap_turns(dict(u["gap"], recovered_text=text), rttm))
    lines.sort(key=lambda x: x[0])
    path = out / f"transcript_speakers_{system}.txt"
    path.write_text("\n".join(f"[{hhmmss(t)}] {spk}: {txt}" for t, spk, txt in lines) + "\n", encoding="utf-8")
    (out / "step1c_segments.json").write_text(json.dumps(
        [{k: v for k, v in u.items() if k != "gap"} for u in units], ensure_ascii=False, indent=1), encoding="utf-8")
    empty = sum(1 for u in units if not u.get(system))
    print(f"{system}: {sum(len(t.split()) for _, _, t in lines)} words in {len(lines)} turns, "
          f"{empty} of {len(units)} clips empty -> {path}")

print(f"{MEETING}: {len(turns)} turns, {len(gaps)} stretches without Canary words, "
      f"{len(pieces)} clips of at most 28 s, {sum(len(x) for _, x in pieces) / sr / 60:.1f} min of audio")

In [ ]:
# 2. Whisper large-v3-turbo-German (loaded as in recognizers/NB_v10_asr.ipynb)
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

MODEL_ID = "primeline/whisper-large-v3-turbo-german"
print(f"=== whisper_turbo_de ({MODEL_ID}) ===")
m = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID, low_cpu_mem_usage=True, use_safetensors=True, **DTYPE_KW).to(DEVICE)
pr = AutoProcessor.from_pretrained(MODEL_ID)
pipe = pipeline("automatic-speech-recognition", model=m, tokenizer=pr.tokenizer,
                feature_extractor=pr.feature_extractor, device=DEVICE, **DTYPE_KW)

texts = []
for k, (_, x) in enumerate(pieces, 1):
    texts.append(pipe({"raw": x, "sampling_rate": sr}, generate_kwargs={"language": "german"})["text"])
    if k % 50 == 0 or k == len(pieces):
        print(f"  {k}/{len(pieces)}", flush=True)
save("whisper_turbo_de", texts)
del pipe, m
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 3. Cohere Transcribe (loaded as in recognizers/NB_v10_asr.ipynb, with its three transformers 5 fixes)
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq

# Fix 1: the remote code declares _keys_to_ignore_on_load_unexpected as a list, but
# transformers 5.15 joins it with a set (list | set raises TypeError). Convert such lists to sets.
import transformers.modeling_utils as _MU
if not getattr(_MU.PreTrainedModel, "_ignore_keys_setshim", False):
    _orig_adjust = _MU.PreTrainedModel._adjust_missing_and_unexpected_keys
    def _adjust_setshim(self, *a, **kw):
        for _attr in ("_keys_to_ignore_on_load_unexpected", "_keys_to_ignore_on_load_missing"):
            _v = getattr(type(self), _attr, None)
            if isinstance(_v, list):
                setattr(type(self), _attr, set(_v))
        return _orig_adjust(self, *a, **kw)
    _MU.PreTrainedModel._adjust_missing_and_unexpected_keys = _adjust_setshim
    _MU.PreTrainedModel._ignore_keys_setshim = True
    print("applied transformers-5 list/set shim")

MODEL_ID = "CohereLabs/cohere-transcribe-03-2026"
print(f"=== cohere ({MODEL_ID}) ===")
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
# .to(DEVICE), not device_map=: device_map needs the accelerate package, which the recorded
# environment (requirements_cluster_canary.txt) does not include. On one GPU both give the same result.
model = AutoModelForSpeechSeq2Seq.from_pretrained(
            MODEL_ID, trust_remote_code=True, **DTYPE_KW).to(DEVICE).eval()
print("loaded class:", type(model).__name__)  # expected: CohereAsrForConditionalGeneration
assert hasattr(model, "transcribe"), "wrong class — AutoModel gives the bare encoder"

# Fix 2: the model's generate() calls super().generate(); in transformers 5 PreTrainedModel
# no longer inherits GenerationMixin, so re-bind the model to a subclass that adds it.
from transformers.generation import GenerationMixin
if not isinstance(model, GenerationMixin):
    model.__class__ = type(type(model).__name__ + "WithGen",
                           (type(model), GenerationMixin), {})
    print("re-bound to", type(model).__name__, "so super().generate() resolves")

# Fix 3: in transformers 5 this model gets no generation_config at init, and generate() needs one.
# Load it from the model repository, or build it from the model config.
from transformers import GenerationConfig
if getattr(model, "generation_config", None) is None:
    try:
        model.generation_config = GenerationConfig.from_pretrained(MODEL_ID)
    except Exception:
        model.generation_config = GenerationConfig.from_model_config(model.config)
    print("attached generation_config")

texts = []
with tempfile.TemporaryDirectory() as tmp:
    paths = []
    for k, (_, x) in enumerate(pieces):
        p = Path(tmp) / f"{k:05d}.wav"
        sf.write(str(p), x, sr)
        paths.append(str(p))
    for b in range(0, len(paths), 16):
        texts.extend(model.transcribe(processor=processor, language="de", audio_files=paths[b:b + 16],
                                      punctuation=True, batch_size=16))
        print(f"  {min(b + 16, len(paths))}/{len(paths)}", flush=True)
save("cohere", texts)
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# 4. Options for step 2
others = [out / "transcript_speakers_cohere.txt", out / "transcript_speakers_whisper_turbo_de.txt"]
missing = [p.name for p in others if not p.exists()]
assert not missing, f"not written yet: {missing}"
print("Add these two options to the protokoll.py command of this meeting:")
print(" \\\n".join(f"  --also-transcript {p.as_posix()}" for p in others))